<a href="https://colab.research.google.com/github/Nova3012k/Deep-learning-2026/blob/main/Multi_layer_Perceptron_for_banking%2C_product_subscription_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Multi layer Perceptron (MLP) Models on Real World Banking Data**


1. Descripción de la librería Scikit-Learn para MLPs
Scikit-Learn es la biblioteca líder en Python para aprendizaje automático, proporcionando herramientas eficientes para el procesamiento de datos y modelos predictivos.

**1.1 Clases y funciones empleadas**

- MLPClassifier: Implementación de la red neuronal que optimiza la función de pérdida mediante el descenso del gradiente.

- LabelEncoder: Clase utilizada para convertir etiquetas categóricas (texto) en valores numéricos.

- StandardScaler: Herramienta para estandarizar las características eliminando la media y escalando a la varianza unitaria.

- accuracy_score: Función para calcular el porcentaje de predicciones correctas.

**1.2 Parámetros**

- hidden_layer_sizes=(13, 10, 2): Define la arquitectura de la red. En este caso, el blog utiliza tres capas ocultas con 13, 10 y 2 neuronas respectivamente.

- max_iter=1000: El número máximo de iteraciones para que el optimizador converja.

# **2. Pipeline de Implementación**

## **2.1 Descripción del problema de clasificación**
El problema consiste en predecir si un cliente bancario contratará un depósito a plazo fijo (variable y) basándose en una campaña de marketing telefónico. Es un problema de clasificación binaria (Sí/No).

### **2.2 Exploración y preprocesamiento del Dataset**
**2.2.1 Descripción de las variables**

Edad (age): Numérica.

Trabajo (job): Categórica (admin, technician, etc.).

Estado Civil (marital): Categórica.

Educación (education): Categórica.

Balance: Saldo promedio anual del cliente.

**2.2.2 Variables independientes y dependiente**

Variable Dependiente (Target): y (¿Suscribió el depósito?).

Variables Independientes (Features): Todas las demás columnas demográficas y de historial de contacto (edad, balance, día, mes, etc.).

**2.2.3 Variables eliminadas y justificación**

Se suele eliminar la variable duration (duración de la llamada).

¿Por qué? Esta variable no se conoce antes de realizar la llamada. Si la duración es 0, entonces y siempre será 'no'. Incluirla genera un modelo "tramposo" que no sirve para predecir antes de hacer la llamada.

**2.2.4 Estandarización: **

¿Por qué y cómo?
Las Redes Neuronales son muy sensibles a la escala. Si una variable tiene valores entre 0-1 (como el éxito previo) y otra entre 0-100,000 (como el balance), los pesos de la red colapsarán.

Cómo: Se resta la media y se divide por la desviación estándar.

In [6]:
# Ejemplo de estandarización
from sklearn.preprocessing import StandardScaler
import numpy as np

datos = np.array([[20], [50], [80]])
sc = StandardScaler()
print("Original:\n", datos)
print("Estandarizado:\n", sc.fit_transform(datos))

Original:
 [[20]
 [50]
 [80]]
Estandarizado:
 [[-1.22474487]
 [ 0.        ]
 [ 1.22474487]]


## **2.3 Model Selection (Arquitectura)**
Se selecciona una Red Neuronal MLP debido a su robustez frente a otros modelos como regresión logística.

Arquitectura: 3 capas ocultas con tamaños de 13, 10 y 2 neuronas.

Flujo: Los datos normalizados entran por la capa de input, se procesan en las capas ocultas y entregan una clasificación binaria.

## **2.4 Model Training**


In [9]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

# Carga de datos
df = pd.read_csv('bank-additional-full.csv', sep=';')

# Preprocesamiento con LabelEncoder
LE = LabelEncoder()
df['job_code'] = LE.fit_transform(df['job'])
df['marital_code'] = LE.fit_transform(df['marital'])
df['education_code'] = LE.fit_transform(df['education'])
df['housing_code'] = LE.fit_transform(df['housing'])
df['loan_code'] = LE.fit_transform(df['loan'])
df['contact_code'] = LE.fit_transform(df['contact'])
df['poutcome_code'] = LE.fit_transform(df['poutcome'])
df['subscribed'] = LE.fit_transform(df['y'])

# Eliminación de columnas no deseadas
df = df.drop(['job','marital','education','housing','loan','contact','poutcome','y','day_of_week','month','default'], axis=1)

# Preparación de datos
X = df.drop('subscribed', axis=1)
y = df['subscribed']
X_train, X_test, y_train, y_test = train_test_split(X, y)

# Estandarización
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Entrenamiento del modelo
mlp = MLPClassifier(hidden_layer_sizes=(13, 10, 2), max_iter=1000)
mlp.fit(X_train, y_train)
print("Modelo entrenado con éxito.")

Modelo entrenado con éxito.


# **2.5 Model Prediction**
Construimos la función para validar patrones específicos:

In [10]:
def prediccion(modelo, scaler_obj, nuevo_patron):
    """
    Función para comprobar que un patrón de entrada se clasifica correctamente.
    """
    # El patrón debe ser una lista de valores numéricos
    patron_scaled = scaler_obj.transform([nuevo_patron])
    resultado = modelo.predict(patron_scaled)

    return "Suscribirá" if resultado[0] == 1 else "No suscribirá"

# Comprobación con un dato del set de prueba
ejemplo = X_test[0]
print(f"Resultado de la predicción: {prediccion(mlp, scaler, ejemplo)}")

Resultado de la predicción: No suscribirá


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


# **2.6 Model Evaluation**
Aplicamos las métricas señaladas en el blog para verificar la robustez del modelo.

In [11]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

predictions = mlp.predict(X_test)

print("MATRIZ DE CONFUSIÓN:")
print(confusion_matrix(y_test, predictions))

print("\nREPORTE DE CLASIFICACIÓN:")
print(classification_report(y_test, predictions))

print(f"\nACCURACY SCORE: {accuracy_score(y_test, predictions):.4f}")

MATRIZ DE CONFUSIÓN:
[[8670  438]
 [ 441  748]]

REPORTE DE CLASIFICACIÓN:
              precision    recall  f1-score   support

           0       0.95      0.95      0.95      9108
           1       0.63      0.63      0.63      1189

    accuracy                           0.91     10297
   macro avg       0.79      0.79      0.79     10297
weighted avg       0.91      0.91      0.91     10297


ACCURACY SCORE: 0.9146
